In [1]:
import ray
from ray.data.llm import vLLMEngineProcessorConfig, build_processor
import torch
from transformers import pipeline
import pandas as pd
from sentence_transformers import SentenceTransformer
from PIL import Image
import os

INFO 08-24 10:11:02 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 08-24 10:11:02 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


W0824 10:11:03.693000 69166 site-packages/torch/utils/cpp_extension.py:118] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


# Transform and Ingest eCom Product Catalog with Ray Data

In this module, we start with a basic product catalog and prepare it for use by...
* applying basic data transformations
* creating embedddings for product descriptions, to support semantic search
* generate extended human-facing catalog copy for each item, using a multimodal model

We'll do this mostly using Ray Data and along the way, we'll discuss
* performance considerations
* using AI to generate and improve our code
* observability
* gotchas/best practices

Our initial catalog is tabular parquet data. Let's inspect it.

In [2]:
cat = ray.data.read_parquet('s3://anyscale-public-materials-use2/ecom/catalog')
cat

2026-08-24 10:11:04,798	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 100.87.9.27:6379...
2026-08-24 10:11:04,835	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-gy6zbz8z924rvbk9ur6wsep7e1.i.anyscaleuserdata.com 
2026-08-24 10:11:04,839	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_88690179c81e4658035d19b6a0c08ab96021bd79.zip' (0.44MiB) to Ray cluster...
2026-08-24 10:11:04,841	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_88690179c81e4658035d19b6a0c08ab96021bd79.zip'.
/home/ray/anaconda3/lib/python3.11/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


shape: (?, 7)
╭─────────────────────────────┬────────────────────┬────────────────────────────┬──────────┬───────────┬──────────────────┬───────────╮
│ manufacturer_product_number ┆ parent_item_number ┆ ecom_vendor_product_number ┆ category ┆ item_name ┆ item_description ┆ price_usd │
│ ---                         ┆ ---                ┆ ---                        ┆ ---      ┆ ---       ┆ ---              ┆ ---       │
│ string                      ┆ string             ┆ string                     ┆ string   ┆ string    ┆ string           ┆ double    │
╰─────────────────────────────┴────────────────────┴────────────────────────────┴──────────┴───────────┴──────────────────┴───────────╯
(Dataset isn't materialized)

In [3]:
cat.count()

2026-08-24 10:11:05,181	INFO logging.py:416 -- Registered dataset logger for dataset dataset_19_0
2026-08-24 10:11:05,207	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_19_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:11:05,207	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_19_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[MapBatches(count_rows)]
2026-08-24 10:11:05,212	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 28.0% of available memory (26.9GiB out of 96.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
2026-08-24 10:11:05,213	INFO __init__.py:56 -- Progress will be logged be

1000

In [4]:
sample_records = cat.take(20)

pd.DataFrame(sample_records)

2026-08-24 10:11:09,468	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-08-24 10:11:09,470	INFO logging.py:416 -- Registered dataset logger for dataset dataset_20_0
2026-08-24 10:11:09,475	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_20_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:11:09,476	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_20_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=20]
2026-08-24 10:11:09,498	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_20_0 =======
2026-08-24 10:11:09,499	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 10:11:09,500	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store
2026-08-24 10:11:09,500	INFO logging_progress.py:181 

,manufacturer_product_number,parent_item_number,ecom_vendor_product_number,category,item_name,item_description,price_usd
0,GLD-N7ONCJT4Q,PARENT-843CELW,VEN-4X3TKQIX7,Housewares,Stainless Steel Mixing Bowl Set (with Lid) (Blue),A nesting set of durable stainless steel bowls...,39.49
1,KNT-JDCXZVZI6,PARENT-SFB1I1F,ECO-MZXPOJ5PI,Sports & Outdoors,"Portable Picnic Blanket (Size M, Gray)",A foldable blanket with a water-resistant back...,58.99
2,LUX-AJ44IB93I,PARENT-SV5X38Y,MKP-QP2SDR1C7,Grocery & Pantry,Rice (Long Grain) (Bold Flavor) (32 oz),Long grain rice that cooks fluffy for bowls an...,13.49
3,ALP-QJIB4Y8UW,,VEN-5RISQZ6FK,Furniture,Mid-Century Coffee Table (Matte Black),A clean-lined coffee table with sturdy legs an...,42.99
4,IVY-GTJRD6HZU,PARENT-9BISMLR,CAT-S50R1KOTA,Apparel,Men's Classic Chino Pants - Moisture-Wicking (...,"Straight-fit chinos with a versatile, office-t...",22.49
5,NXT-FIX63HU3K,PARENT-JBPHZ9R,HUB-9RJD5G2UI,Grocery & Pantry,Tea Sampler Box (Bold Flavor) (16 oz),A selection of teas with a range of flavors an...,5.00
6,OPL-WWX6VB0BN,PARENT-5B4ZGDO,ECO-L3QH1EW2K,Books,Modern Fiction Collection - Illustrated Editio...,A set of short stories with vivid characters a...,7.49
7,CRS-WBY1CLKZS,PARENT-C6XJX6U,HUB-9H57UHCFF,Housewares,Ceramic Coffee Mug (Family Size) (Beige),A comfy ceramic mug with a smooth glaze for yo...,34.00
8,FWD-5PNRDV7Q1,PARENT-OBT55AO,VEN-J1ZZX0LXR,Apparel,"Unisex Graphic Tee - Moisture-Wicking (Size L,...","A breathable cotton-blend tee with a clean, mo...",87.49
9,IVY-EQER83UUR,PARENT-UQQMI40,MKP-T6383U8S0,Sports & Outdoors,"Fishing Tackle Box (Size L, Black)",A portable tackle box with compartments for ho...,31.99


In [5]:
sample_records[0]

{'manufacturer_product_number': 'GLD-N7ONCJT4Q',
 'parent_item_number': 'PARENT-843CELW',
 'ecom_vendor_product_number': 'VEN-4X3TKQIX7',
 'category': 'Housewares',
 'item_name': 'Stainless Steel Mixing Bowl Set (with Lid) (Blue)',
 'item_description': 'A nesting set of durable stainless steel bowls for everyday prep. Easy to wipe clean and store between uses.',
 'price_usd': 39.49}

Each record contains manufacturer product ID, our (eCom vendor) product ID, category, name, description, and price.

Some items also contain a parent ID, which links variants of the same product (e.g., blue vs. red blanket or medium vs. large shirt)

## Ingest and basic transformation

It may be convenient to shorten some of the original field names, and we can see a basic pattern of applying transformations to a Dataset.

Each transformation produces a new Dataset, so operations can be chained together to form a dataprocessing pipeline. 

Datasets are streaming, lazy-evaluated abstractions for data collections, so
1. At the end of a pipeline, we need to "send" this pipeline somewhere, e.g., stable storage (filesystem, blob store), a database, etc.
2. Creating lots of new, intermediate Datasets is inexpensive since the Dataset itself does not contain nor eagerly operate on the actual data.

In [6]:
renames = {
    'manufacturer_product_number' : 'mfg_item_id',
    'parent_item_number' : 'parent_id',
    'ecom_vendor_product_number' : 'item_id',
    'category' : 'cat',
    'item_name' : 'name',
    'item_description' : 'desc',
    'price_usd' : 'price'
}
cat.rename_columns(renames).write_parquet('/mnt/cluster_storage/catalog', mode=ray.data.SaveMode.OVERWRITE)

2026-08-24 10:11:16,619	INFO logging.py:416 -- Registered dataset logger for dataset dataset_23_0
2026-08-24 10:11:16,623	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_23_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:11:16,624	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_23_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> TaskPoolMapOperator[Write]
2026-08-24 10:11:16,642	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_23_0 =======
2026-08-24 10:11:16,643	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 10:11:16,644	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store
2026-08-24 10:11:16,644	INFO logging_progress.py:181 -- 
2026-08-24 10:11:16,645	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-08-24 10:11:16,646	INFO logging_progress.py:233 --   Tasks: 1; Actors

Note that we read the source dataset from `S3` and wrote it to `/mnt/cluster_storage` (global to our cluster, but not visible elsewhere).

In [7]:
! ls /mnt

cluster_storage  local_storage	shared_storage	user_storage


*Storage scopes* (shared per-org storage, per-user storage, fast machine-local storage, and a pre-configured cloud blobstore bucket) provide easy best-practices options for data whenever using Anyscale.

Earlier, we applied `take` to get a list of records. Frequently, we operate on batches of records, so we can `take_batch` as well

In [8]:
sample_batch = ray.data.read_parquet('/mnt/cluster_storage/catalog/').take_batch(3)
sample_batch

2026-08-24 10:12:47,238	INFO logging.py:416 -- Registered dataset logger for dataset dataset_26_0
2026-08-24 10:12:47,244	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_26_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:12:47,244	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_26_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=3]
2026-08-24 10:12:47,263	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_26_0 =======
2026-08-24 10:12:47,264	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 10:12:47,265	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store
2026-08-24 10:12:47,266	INFO logging_progress.py:181 -- 
2026-08-24 10:12:47,267	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-08-24 10:12:47,267	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0;

{'mfg_item_id': array(['GLD-N7ONCJT4Q', 'KNT-JDCXZVZI6', 'LUX-AJ44IB93I'], dtype=object),
 'parent_id': array(['PARENT-843CELW', 'PARENT-SFB1I1F', 'PARENT-SV5X38Y'], dtype=object),
 'item_id': array(['VEN-4X3TKQIX7', 'ECO-MZXPOJ5PI', 'MKP-QP2SDR1C7'], dtype=object),
 'cat': array(['Housewares', 'Sports & Outdoors', 'Grocery & Pantry'],
       dtype=object),
 'name': array(['Stainless Steel Mixing Bowl Set (with Lid) (Blue)',
        'Portable Picnic Blanket (Size M, Gray)',
        'Rice (Long Grain) (Bold Flavor) (32 oz)'], dtype=object),
 'desc': array(['A nesting set of durable stainless steel bowls for everyday prep. Easy to wipe clean and store between uses.',
        'A foldable blanket with a water-resistant backing for parks. Lightweight and packable for travel.',
        'Long grain rice that cooks fluffy for bowls and sides. Packed for freshness with resealable packaging.'],
       dtype=object),
 'price': array([39.49, 58.99, 13.49])}

Note the form of the batch

In [9]:
type(sample_batch)

dict

In [10]:
type(sample_batch['price'])

numpy.ndarray

The Dataset API docs detail the transformations, which include projection, filtering, joins, and more.

E.g., we can filter for the "Housewares" category.

In [11]:
ray.data.read_parquet('/mnt/cluster_storage/catalog/').filter(lambda row:row['cat']=='Housewares').take(3)

/home/ray/anaconda3/lib/python3.11/site-packages/ray/data/dataset.py:1633: UserWarning: Use 'expr' instead of 'fn' when possible for performant filters.
  warnings.warn(
2026-08-24 10:12:55,424	INFO logging.py:416 -- Registered dataset logger for dataset dataset_29_0
2026-08-24 10:12:55,429	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_29_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:12:55,430	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_29_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> TaskPoolMapOperator[Filter(<lambda>)] -> LimitOperator[limit=3]
2026-08-24 10:12:55,457	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_29_0 =======
2026-08-24 10:12:55,458	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 10:12:55,459	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object s

[{'mfg_item_id': 'GLD-N7ONCJT4Q',
  'parent_id': 'PARENT-843CELW',
  'item_id': 'VEN-4X3TKQIX7',
  'cat': 'Housewares',
  'name': 'Stainless Steel Mixing Bowl Set (with Lid) (Blue)',
  'desc': 'A nesting set of durable stainless steel bowls for everyday prep. Easy to wipe clean and store between uses.',
  'price': 39.49},
 {'mfg_item_id': 'CRS-WBY1CLKZS',
  'parent_id': 'PARENT-C6XJX6U',
  'item_id': 'HUB-9H57UHCFF',
  'cat': 'Housewares',
  'name': 'Ceramic Coffee Mug (Family Size) (Beige)',
  'desc': 'A comfy ceramic mug with a smooth glaze for your daily pour. Easy to wipe clean and store between uses.',
  'price': 34.0},
 {'mfg_item_id': 'OPL-NSFVLJFH1',
  'parent_id': 'PARENT-7QUVY65',
  'item_id': 'HUB-9KUKVNM3I',
  'cat': 'Housewares',
  'name': 'Bamboo Cutting Board (Gray)',
  'desc': 'A sturdy bamboo board that’s gentle on knives and easy to rinse clean. Easy to wipe clean and store between uses.',
  'price': 6.0}]

As of 2026, Ray Data has integrated a query optimizer and has begun to implement a planner that provides logical and physical optimizations.

In order to optimize, though, Ray needs to have legible, reified operands.

In the previous example, the `filter` operator is explicit enough (we coded it, and Python can parse it) but the `cat` column and the `Housewares` value are opaque: they're hidden inside the lambda. Since a lambda can contain (nearly) arbitrary code, finding the semantic operands in that code is not a preferred approach.

Instead, Ray Data is now implementing an expression language to support optimizations (similar to many other data processing frameworks). We (or our AI agents) write the expressions, which are then legible to the optimizers.

In [12]:
from ray.data.expressions import col

ray.data.read_parquet('/mnt/cluster_storage/catalog/').filter(expr=col('cat') == 'Housewares').take(3)

2026-08-24 10:13:02,540	INFO logging.py:416 -- Registered dataset logger for dataset dataset_32_0
2026-08-24 10:13:02,545	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_32_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:13:02,546	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_32_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=3]
2026-08-24 10:13:02,565	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_32_0 =======
2026-08-24 10:13:02,566	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 10:13:02,567	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store
2026-08-24 10:13:02,568	INFO logging_progress.py:181 -- 
2026-08-24 10:13:02,569	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-08-24 10:13:02,570	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0;

[{'mfg_item_id': 'GLD-N7ONCJT4Q',
  'parent_id': 'PARENT-843CELW',
  'item_id': 'VEN-4X3TKQIX7',
  'cat': 'Housewares',
  'name': 'Stainless Steel Mixing Bowl Set (with Lid) (Blue)',
  'desc': 'A nesting set of durable stainless steel bowls for everyday prep. Easy to wipe clean and store between uses.',
  'price': 39.49},
 {'mfg_item_id': 'CRS-WBY1CLKZS',
  'parent_id': 'PARENT-C6XJX6U',
  'item_id': 'HUB-9H57UHCFF',
  'cat': 'Housewares',
  'name': 'Ceramic Coffee Mug (Family Size) (Beige)',
  'desc': 'A comfy ceramic mug with a smooth glaze for your daily pour. Easy to wipe clean and store between uses.',
  'price': 34.0},
 {'mfg_item_id': 'OPL-NSFVLJFH1',
  'parent_id': 'PARENT-7QUVY65',
  'item_id': 'HUB-9KUKVNM3I',
  'cat': 'Housewares',
  'name': 'Bamboo Cutting Board (Gray)',
  'desc': 'A sturdy bamboo board that’s gentle on knives and easy to rinse clean. Easy to wipe clean and store between uses.',
  'price': 6.0}]

With a format like `parquet`, which is columnar, an explicit filter on a column value (as above) can potentially enable a predicate pushdown optimization, allowing less data to be read from the datasource.

## Generating embeddings for semantic search

Let's start with some sample code for an embedding model we want to use. The sample code -- perhaps from other repos at our company, from Huggingface, or from Google (the model provider in this instance) is not Ray enabled.

We will convert it for Ray. It's not a lot of work, but, later, we'll see how to use AI to do most of this work for even higher productivity.

In [13]:
! aws s3 sync s3://anyscale-public-materials-use2/ecom/hf_cache /mnt/cluster_storage/hf_cache

download: s3://anyscale-public-materials-use2/ecom/hf_cache/.locks/models--google--embeddinggemma-300m/1299c11d7cf632ef3b4e11937501358ada021bbdf7c47638d13c0ee982f2e79c.lock to ../../../mnt/cluster_storage/hf_cache/.locks/models--google--embeddinggemma-300m/1299c11d7cf632ef3b4e11937501358ada021bbdf7c47638d13c0ee982f2e79c.lock
download: s3://anyscale-public-materials-use2/ecom/hf_cache/.locks/models--google--embeddinggemma-300m/52373fe24473b1aa44333d318f578ae6bf04b49b.lock to ../../../mnt/cluster_storage/hf_cache/.locks/models--google--embeddinggemma-300m/52373fe24473b1aa44333d318f578ae6bf04b49b.lock
download: s3://anyscale-public-materials-use2/ecom/hf_cache/.locks/models--google--embeddinggemma-300m/33786d29ae85b05bc37772cd10876d66afbe905e.lock to ../../../mnt/cluster_storage/hf_cache/.locks/models--google--embeddinggemma-300m/33786d29ae85b05bc37772cd10876d66afbe905e.lock
download: s3://anyscale-public-materials-use2/ecom/hf_cache/.locks/models--google--embeddinggemma-300m/3552ab1b79dc

In [14]:
! aws s3 sync s3://anyscale-public-materials-use2/ecom/catalog_images /mnt/cluster_storage/catalog_images

download: s3://anyscale-public-materials-use2/ecom/catalog_images/AMZ-0CRR6WRX9.png to ../../../mnt/cluster_storage/catalog_images/AMZ-0CRR6WRX9.png
download: s3://anyscale-public-materials-use2/ecom/catalog_images/AMZ-003C77I0C.png to ../../../mnt/cluster_storage/catalog_images/AMZ-003C77I0C.png
download: s3://anyscale-public-materials-use2/ecom/catalog_images/AMZ-0DMS3N759.png to ../../../mnt/cluster_storage/catalog_images/AMZ-0DMS3N759.png
download: s3://anyscale-public-materials-use2/ecom/catalog_images/AMZ-1KM29H3W3.png to ../../../mnt/cluster_storage/catalog_images/AMZ-1KM29H3W3.png
download: s3://anyscale-public-materials-use2/ecom/catalog_images/AMZ-0W4WKMPDG.png to ../../../mnt/cluster_storage/catalog_images/AMZ-0W4WKMPDG.png
download: s3://anyscale-public-materials-use2/ecom/catalog_images/AMZ-3ACE7SCZS.png to ../../../mnt/cluster_storage/catalog_images/AMZ-3ACE7SCZS.png
download: s3://anyscale-public-materials-use2/ecom/catalog_images/AMZ-1W465LYCK.png to ../../../mnt/cluste

In [15]:
cached_embedding_model = "/mnt/cluster_storage/hf_cache/models--google--embeddinggemma-300m/snapshots/57c266a740f537b4dc058e1b0cda161fd15afa75"
model = SentenceTransformer(cached_embedding_model)

sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day"
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)

similarities

tensor([[1.0000, 0.8078, 0.9839, 0.5595],
        [0.8078, 1.0000, 0.7960, 0.5486],
        [0.9839, 0.7960, 1.0000, 0.5417],
        [0.5595, 0.5486, 0.5417, 1.0000]])

In [16]:
type(embeddings)

numpy.ndarray

The standard pattern for generating embeddings -- or applying any other batch inference to a large dataset -- is to use the Dataset `map_batches` operator together with instances of a Ray Actor class that maintains the model state and performs the inference.

To implement this pattern, we do the following steps:

1. Create the class as a plain old Python class. Ray will take care of launching it on relevant workers as an Actor and managing instances.

2. Apply the `map_batches` Dataset operator, with some details about the class to use and, optionally, devices (e.g., GPU), batch sizes, scaling limits, etc.

The basic class might look like this:

In [17]:
class AddEmbedding():
    def __init__(self, model_id):
        self.model = SentenceTransformer(model_id)

    def __call__(self, batch, source, dest):
        inputs = batch[source]
        outputs = self.model.encode(inputs)
        batch[dest] = outputs
        return batch

In that class, the constructor handles acquiring and holding large pieces of state (the model itself), while the `__call__` method accepts a batch of Dataset records (by default, in the Python `dict` format we saw earlier) and transforms that batch -- in this case, by adding a new field with the generated vectors.

When we call `map_batches`, we'll need to provide the class name (`AddEmbedding`) and -- because our class expects some additional values beyond just the data batches, we'll need to provide these "extra values" for our class' constructor and `__call__` methods. This pattern allows for cleaner separation of concerns (e.g., changing model name).

So a pipeline might look like this:

```python
ray.data.read_parquet('/mnt/cluster_storage/catalog/') \
    .map_batches(AddEmbedding, compute=ray.data.ActorPoolStrategy(size=2), fn_constructor_args=["google/embeddinggemma-300m"], fn_args=['desc', 'desc_emb']) \
    .take(3)
```

__Developer Tips__

> If we're just getting started with this pipeline, we might want to test it out on a small number of records. To do that, we can 
> 1. Add a `limit` operator
>
> and 
>
> 2. Compute and cache a small bit of the dataset *before* trying our new transformation, to avoid Ray computing an entire Dataset block (which might be much larger than our `limit`) each time we try to test our code.

To compute and cache a Dataset, use the `materialize` operator. This operator places the resulting data blocks in the Ray cluster's Object Store (either in memory across the cluster workers, or, if the data is too large, spilled to disk across the workers).

In [18]:
cached_test_dataset = ray.data.read_parquet('/mnt/cluster_storage/catalog/').limit(10).materialize()
cached_test_dataset

2026-08-24 10:21:49,380	INFO logging.py:416 -- Registered dataset logger for dataset dataset_35_0
2026-08-24 10:21:49,384	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_35_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:21:49,385	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_35_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=10]
2026-08-24 10:21:49,404	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_35_0 =======
2026-08-24 10:21:49,405	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 10:21:49,406	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store
2026-08-24 10:21:49,407	INFO logging_progress.py:181 -- 
2026-08-24 10:21:49,407	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-08-24 10:21:49,409	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0

shape: (10, 7)
╭───────────────┬────────────────┬───────────────┬───────────────────┬──────────────────────────────────────────┬──────────────────────────────────────────┬────────╮
│ mfg_item_id   ┆ parent_id      ┆ item_id       ┆ cat               ┆ name                                     ┆ desc                                     ┆ price  │
│ ---           ┆ ---            ┆ ---           ┆ ---               ┆ ---                                      ┆ ---                                      ┆ ---    │
│ string        ┆ string         ┆ string        ┆ string            ┆ string                                   ┆ string                                   ┆ double │
╞═══════════════╪════════════════╪═══════════════╪═══════════════════╪══════════════════════════════════════════╪══════════════════════════════════════════╪════════╡
│ GLD-N7ONCJT4Q ┆ PARENT-843CELW ┆ VEN-4X3TKQIX7 ┆ Housewares        ┆ Stainless Steel Mixing Bowl Set (with L… ┆ A nesting set of durable stainless stee… 

Now we can test our embedding implementation

In [19]:
test = cached_test_dataset \
    .map_batches(AddEmbedding, compute=ray.data.ActorPoolStrategy(size=2), fn_constructor_args=[cached_embedding_model], fn_args=['desc', 'desc_emb']) \
    .take(3)

2026-08-24 10:21:49,811	INFO logging.py:416 -- Registered dataset logger for dataset dataset_38_0
2026-08-24 10:21:49,814	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_38_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:21:49,815	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_38_0: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(AddEmbedding)] -> LimitOperator[limit=3]
{"asctime":"2026-08-24 10:21:49,842","levelname":"E","message":"Actor with class name: 'MapWorker(MapBatches(AddEmbedding))' and ID: '892f95642f52e32885136c8904000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.","filename":"core_worker.cc","lineno":2194}
2026-08-24 10:21:49,973	INFO logging_progress.py:174 -- ======= Running Datase

In [20]:
type(test[0]['desc_emb'])

numpy.ndarray

In [21]:
test[0]['desc_emb'].shape

(768,)

Let's compute the embeddings for all of our product descriptions. Since we're currently using just CPU for model inference, we can let Ray autoscale the number of actors based on our cluster resources and the relative speeds of different parts of our pipeline (and potentially the resource usage of other operations simultaneously running on our Ray cluster).

Once we start the pipeline processing, open the Ray Dashboard (in a separate tab) and observe the Actors tab.

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/catalog/') \
    .map_batches(AddEmbedding, fn_constructor_args=[cached_embedding_model], fn_args=['desc', 'desc_emb']) \
    .write_parquet('/mnt/cluster_storage/cat_with_embeddings', mode=ray.data.SaveMode.OVERWRITE)

2026-08-24 10:22:10,607	INFO logging.py:416 -- Registered dataset logger for dataset dataset_42_0
2026-08-24 10:22:10,614	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_42_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 10:22:10,614	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_42_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(AddEmbedding)] -> TaskPoolMapOperator[Write]
2026-08-24 10:22:10,757	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_42_0 =======
2026-08-24 10:22:10,758	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 10:22:10,759	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store (pending: 1 CPU)
2026-08-24 10:22:10,761	INFO logging_progress.py:181 -- 
2026-08-24 10:22:10,761	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-08

The pipeline works!

Let's look at making it faster. This run takes about 1.5 minutes.

The first issue to identify is that, despite our "large" cluster, Ray is only running one Actor to process all of this data. The 1 Actor can be seen in the log output as well as in the Ray Dashboard.

__Parallelism and Ray Dataset Blocks__

Since datasets may be arbitrarily large, and Ray Data processes them in a streaming fashion, there is a core unit of data which is handled by Ray to make up the stream.

This unit is a block. While *user code* (e.g., your AI inference or other business logic) can specify arbitrary *batch sizes*, Ray Data's block sizes are independent of those values and are typically much larger (e.g., 100MB to 1GB). Ray calculates a block size automatically although you can override that size if needed. Ray can also combine small files (and in some cases split larger files) to generate blocks.

In our example, though, since our catalog is small, Ray is only creating one block and -- with just one block to process -- it only runs one Actor.

We can verify the number of blocks if we compute the source dataset

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/catalog/').materialize()

We can ask Ray to split our data into more blocks using the `repartition` operator.

After starting this pipeline, observe the Actor count in the logs and on the dashboard.

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/catalog/') \
    .repartition(8) \
    .map_batches(AddEmbedding, fn_constructor_args=[cached_embedding_model], fn_args=['desc', 'desc_emb']) \
    .write_parquet('/mnt/cluster_storage/cat_with_embeddings', mode=ray.data.SaveMode.OVERWRITE)

The parallelism improved our performance significantly. Depending on the availability of GPUs and the relative priority of this workload, we may want to benchmark with an accelerator as well.

__Developer Tip__

> If you are developing in an interactive environment like a notebook, if a Ray Data pipeline fails (or if you interrupt the execution), existing Actors may remain running and holding on to resources in the cluster. You can kill them all by running `ray.shutdown()` -- in the Anyscale environment, this will *not* shutdown your physical cluster but it will cause your next operations to re-initialize Ray in a clean manner.

To use an accelerator for our embeddings, we need to

1. Modify the Actor class code (if needed) to leverage an accelerator
2. Adjust our `map_batches` call as follows:
    1. Specify `num_gpus` per Actor
    1. Specify `batch_size` so that we use our GPU RAM but don't exceed it
    1. Provide a `compute` strategy -- Ray won't autoscale GPU-based actors on its own

In [ ]:
class AddEmbedding():
    def __init__(self, model_id):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_id).to(self.device)

    def __call__(self, batch, source, dest):
        inputs = batch[source]
        outputs = self.model.encode(inputs)
        batch[dest] = outputs
        return batch

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/catalog/').repartition(4) \
    .map_batches(AddEmbedding, num_gpus=1, batch_size=64,
                 compute=ray.data.ActorPoolStrategy(size=2), fn_constructor_args=[cached_embedding_model], fn_args=['desc', 'desc_emb']) \
    .write_parquet('/mnt/cluster_storage/cat_with_embeddings', mode=ray.data.SaveMode.OVERWRITE)

Unsurprisingly, running batches on a GPU is much faster (about 8 seconds here).

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/cat_with_embeddings/').take(1)

## Creating catalog product copy using a multimodal AI model

We want to create a longer product description for each item, based on the item's description and its picture.

The longer descriptions will be great for SEO, enticing our customers to consider a product purchase, etc.

Let's start with a basic image + text -> text model and a Huggingface usage code sample

In [ ]:
candy_demo_image_path = '/mnt/cluster_storage/hf_cache/candy.jpg'

Image.open(candy_demo_image_path).resize((800,600))

In [ ]:
cached_multimodal_model = '/mnt/cluster_storage/hf_cache/models--google--gemma-3-4b-it/snapshots/093f9f388b31de276ce2de164bdc2081324b9767'

pipe = pipeline("image-text-to-text", model=cached_multimodal_model)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "path": candy_demo_image_path},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages, max_new_tokens=1024)

That "hello world" establishes that we can successfully run the model.

We could try to straight to adding suitable code to our Ray Data pipeline ... but if it doesn't work optimally, using (and re-running) a full pipeline can be timeconsuming and cumbersome.

Let's take an incremental step by running this code in a Ray Actor. Running in an Actor allows us easily to accomplish several goals:
1. Ensure the code runs properly on our chosen accelerator type (the sample above defaulted to CPU)
2. Test/experiment with prompts to ensure our prompting works and our output matches our desired format, style, length, etc.
3. Evaluate memory consumption when running on a GPU node, so that we can tune a batch size.

In [ ]:
system_prompt = '''You are a helpful assistant. Given a product name and description, and an image of that product, 
please create a more descriptive and attractive blurb for the product, capturing elements from the image and 
suitable for use in an ecommerce website where that product is for sale. Output a single suggestion or option, 
and do not include any additional conversational language or discussion.'''

@ray.remote(num_gpus=1)
class IT2T():
    def __init__(self):
        self.pipe = pipeline("image-text-to-text", model=cached_multimodal_model, device='cuda')
        
    def prompt(self, image_key="path", image_val='/mnt/cluster_storage/hf_cache/candy.jpg', text="What animal is on the candy?"):
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": system_prompt}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", image_key: image_val},
                    {"type": "text", "text": text}
                ]
            },
        ]
        return self.pipe(text=messages, max_new_tokens=1024)

`@ray.remote` and `num_gpus=1` tell Ray to make this an Actor class and only to schedule it where a GPU is available. It will then hold that amount of GPU during its lifetime.

We tell Ray to create an instance of the Actor somewhere in the cluster, and to give us a handle to that Actor, with the following code:

In [ ]:
it2t = IT2T.remote()

To invoke the `prompt` method on the Actor instance, we add `remote()`:

In [ ]:
ref = it2t.prompt.remote()
ref

Remote calls return ObjectRef (a combination of a remote pointer and a promise).

If we want to block (if necessary) and retrieve a local copy of the result, we can use `ray.get`

In [ ]:
ray.get(ref)

That looks like it's in the neighborhood of what we want. Let's test with a real product and image from our catalog.

In [ ]:
product_desc = 'A nesting set of durable stainless steel bowls for everyday prep. Easy to wipe clean and store between uses.'
product_id = 'VEN-4X3TKQIX7'
image_path = f'/mnt/cluster_storage/catalog_images/{product_id}.png'

In [ ]:
Image.open(image_path)

In [ ]:
ref = it2t.prompt.remote('path', image_path, product_desc)

In [ ]:
ray.get(ref)

> __Useful Exercise__ Refactor this code to accept a batch of data and pass the batch to the Huggingface pipeline. Now you can use the actor to test for the maximum reliable batch size.

As long as we have a handle to our Actor (`it2t`), we might use it ... so Ray will keep it available.

Since the notebook's scope lasts forever, we can delete or kill this Actor to release its GPU.

In [ ]:
ray.kill(it2t)

### Using AI to convert our Huggingface "hello world" into Ray Data code

Let's see if Claude Opus 4.6 can help us write Ray code.

We'll give Claude Code the "hello world" snippet, the schema of our Ray Dataset, and a description of our task.

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/cat_with_embeddings/').schema()

Here's a quick and dirty prompt -- it's intentionally a bit sloppy (e.g., copy/paste of the schema):

```python
I have some basic huggingface code:

pipe = pipeline("image-text-to-text", model="google/gemma-3-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages, max_new_tokens=1024)
```
```
I want to adapt this code in the following way:

1. I have a Ray Data dataset of ecom products with the following schema: 

Column       Type ------       ---- mfg_item_id  string parent_id    string item_id      string cat          string name         string desc         string price        double desc_emb     ArrowTensorTypeV2(shape=(768,), dtype=float)

2. I want to run the dataset through this AI model where the model receives the record's "desc" (description) field and an image located under /shared_path/{item_id}.png, and the model generates an extended description of the product suitable for an ecom site, in a new column called "cat_desc"

3. write the results to parquet or a database

Can you show me Ray code which will accomplish the task?
```

__Claude produces the following code__:

```python
import ray
from ray.data import Dataset
from transformers import pipeline
from PIL import Image
import os

# --- Configuration ---
MODEL_NAME = "google/gemma-3-4b-it"
IMAGE_DIR = "/shared_path"
OUTPUT_PATH = "/output/ecom_products_with_cat_desc.parquet"
BATCH_SIZE = 4  # Adjust based on GPU memory

PROMPT_TEMPLATE = (
    "You are an expert ecommerce copywriter. Given the product image and its "
    "current description, write an extended, compelling product description "
    "suitable for an ecommerce website. Be descriptive, highlight key features, "
    "and make it appealing to buyers.\n\n"
    "Current description: {desc}\n\n"
    "Write the extended product description:"
)


class VLMPredictor:
    """Ray actor-based predictor that loads the model once per worker."""

    def __init__(self):
        self.pipe = pipeline(
            "image-text-to-text",
            model=MODEL_NAME,
            device="cuda:0",  # Use "cpu" if no GPU
        )

    def __call__(self, batch: dict) -> dict:
        results = []

        for item_id, desc in zip(batch["item_id"], batch["desc"]):
            # Load the product image
            image_path = os.path.join(IMAGE_DIR, f"{item_id}.png")

            content = []
            if os.path.exists(image_path):
                content.append({"type": "image", "image": Image.open(image_path).convert("RGB")})

            content.append({
                "type": "text",
                "text": PROMPT_TEMPLATE.format(desc=desc or "No description provided."),
            })

            messages = [{"role": "user", "content": content}]

            try:
                output = self.pipe(text=messages, max_new_tokens=1024)
                generated = output[0]["generated_text"][-1]["content"]
            except Exception as e:
                generated = f"Error generating description: {e}"

            results.append(generated)

        batch["cat_desc"] = results
        return batch


def main():
    ray.init()

    # 1. Load your existing Ray dataset (adjust source as needed)
    ds = ray.data.read_parquet("/input/ecom_products.parquet")

    # 2. Run inference using map_batches with a class-based predictor
    #    - concurrency: number of model replicas (one per GPU)
    #    - batch_size: rows per batch sent to each replica
    #    - num_gpus: GPUs allocated per replica
    ds = ds.map_batches(
        VLMPredictor,
        batch_size=BATCH_SIZE,
        concurrency=1,          # Increase if you have multiple GPUs
        num_gpus=1,             # GPUs per worker; set to 0 for CPU
    )

    # 3. Write results to parquet
    ds.write_parquet(OUTPUT_PATH)
    print(f"Done! Results written to {OUTPUT_PATH}")

    # Alternative: write to a database via custom function
    # for batch in ds.iter_batches(batch_size=100, batch_format="pandas"):
    #     batch.to_sql("products", con=engine, if_exists="append", index=False)


if __name__ == "__main__":
    main()
```

__That looks pretty good!__

Let's refactor a tiny bit:
* swap in our cached model path
* remove the `main` script
* use our known-good system prompt
* pass image by path
* implement batching for the HF pipeline

and test it.

In [ ]:
MODEL_NAME = cached_multimodal_model
IMAGE_DIR = "/mnt/cluster_storage/catalog_images/"
OUTPUT_PATH = "/mnt/cluster_storage/scratch/ecom_products_with_cat_desc.parquet"
BATCH_SIZE = 8  # Adjust based on GPU memory

system_prompt = '''You are a helpful assistant. Given a product name and description, and an image of that product, 
please create a more descriptive and attractive blurb for the product, 
capturing elements from the image and suitable for use in an ecommerce website where that product is for sale.
Output a single suggestion or option, and do not include any additional conversational language or discussion.
'''

class VLMPredictor:
    def __init__(self):
        self.pipe = pipeline("image-text-to-text", model=MODEL_NAME, device="cuda:0")

    def __call__(self, batch: dict) -> dict:
        results = []
        messages = []

        for item_id, desc in zip(batch["item_id"], batch["desc"]):
            image_path = os.path.join(IMAGE_DIR, f"{item_id}.png")
            content = []
            content.append({"type": "image", "path": image_path})
            content.append({"type": "text", "text": desc})

            messages.append([
                { "role": "system", "content": [{"type": "text", "text": system_prompt}] },
                {"role": "user", "content": content}
            ])
            
        outputs = self.pipe(text=messages, max_new_tokens=1024, batch_size=len(messages))          
        batch["cat_desc"] = [out[0]["generated_text"][-1]["content"] for out in outputs]
        return batch

First, since this might take a long time to run -- or might fail in odd ways and require troubleshooting -- let's make a tiny version of our dataset.

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/cat_with_embeddings/').limit(64).write_parquet('/mnt/cluster_storage/scratch/tiny_sample', mode=ray.data.SaveMode.OVERWRITE)

Now we'll run our code and tiny dataset on our cluster and see what happens.

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/scratch/tiny_sample/').repartition(4
    ).map_batches(
        VLMPredictor,
        batch_size=BATCH_SIZE,
        concurrency=2,          # Increase if you have multiple GPUs
        num_gpus=1,             # GPUs per worker; set to 0 for CPU
    ).write_parquet(OUTPUT_PATH, mode=ray.data.SaveMode.OVERWRITE)

Let's take a look

In [ ]:
[(item['desc'], item['cat_desc']) for item in ray.data.read_parquet(OUTPUT_PATH).take(5)]

Overall, looks good. We need statistical evals and checks for, e.g., text in the description matching text in the item or photo, but from a data engineering perspective, this runs.

However... note the warning at the top of the batch processing log output about `concurrency` being deprecated. 

Let's ask Claude:

> `Does this code follow the API for the latest version of Ray? if not, please update`

Claude responds (in part):
    
```The concurrency argument is deprecated, and you should use the compute argument instead. Ray The correct approach is compute=ray.data.ActorPoolStrategy(size=n) for a fixed pool of n workers.```

If our AI tool were actually managing our code, we could test or check that the corrections are implemented. Here, we can make the change ourselves.

In [ ]:
ray.data.read_parquet('/mnt/cluster_storage/scratch/tiny_sample/').repartition(4
    ).map_batches(
        VLMPredictor,
        batch_size=BATCH_SIZE,
        compute=ray.data.ActorPoolStrategy(size=2),
        num_gpus=1,
    ).write_parquet(OUTPUT_PATH, mode=ray.data.SaveMode.OVERWRITE)

> <h2 style='color:blue;'>Instructor Demo: Implementing the pipeline as an Anyscale Job</h2>

---

## Accelerating and simplifying inference with RayLLM and vLLM

1. Configure the vLLM engine that Ray Data will run on each GPU replica

In [ ]:
config = vLLMEngineProcessorConfig(
    model_source=cached_multimodal_model,
    engine_kwargs=dict(
        max_model_len=8192,
        limit_mm_per_prompt={"image": 1},   # one image per prompt
        # tensor_parallel_size=1,           # raise to shard one model across >1 GPU
    ),
    batch_size=8,        # rows per batch handed to the engine
    concurrency=2,       # number of engine replicas (≈ GPUs used)
    # accelerator_type="L4",  # optionally pin a GPU type
)

2. Build the processor: preprocess shapes the chat request, postprocess names the output


In [ ]:
processor = build_processor(
    config,
    preprocess=lambda row: dict(
        messages=[
            { "role": "system", "content": [{"type": "text", "text": system_prompt}] },
            { "role": "user", "content": [
                {"type": "text", "text": row["desc"]},
                {"type": "image", "path": os.path.join(IMAGE_DIR, row["item_id"]+".png") }
            ]},
        ],
        sampling_params=dict(temperature=0.3, max_tokens=1024),
    ),
    postprocess=lambda row: dict(
        cat_desc=row["generated_text"],   # the model's extended blurb
        item_id=row["item_id"],
        desc=row["desc"],
    ),
)

3. Apply to the dataset — same source the notebook already produced

In [ ]:
ds = ray.data.read_parquet('/mnt/cluster_storage/scratch/tiny_sample/').repartition(4)
ds = processor(ds)
ds.write_parquet('/mnt/cluster_storage/scratch/tiny_sample_vllm', mode=ray.data.SaveMode.OVERWRITE)

Check output

In [ ]:
[(item['desc'], item['cat_desc']) for item in ray.data.read_parquet('/mnt/cluster_storage/scratch/tiny_sample_vllm').take(5)]

### Stream Direct to GPU with RunAI

In [ ]:
config = vLLMEngineProcessorConfig(
    model_source=cached_multimodal_model,
    engine_kwargs=dict(
        load_format="runai_streamer",
        max_model_len=8192,
        limit_mm_per_prompt={"image": 1},   # one image per prompt
    ),
    batch_size=8,        # rows per batch handed to the engine
    concurrency=2,       # number of engine replicas (≈ GPUs used)
)

processor = build_processor(
    config,
    preprocess=lambda row: dict(
        messages=[
            { "role": "system", "content": [{"type": "text", "text": system_prompt}] },
            { "role": "user", "content": [
                {"type": "text", "text": row["desc"]},
                {"type": "image", "path": os.path.join(IMAGE_DIR, row["item_id"]+".png") }
            ]},
        ],
        sampling_params=dict(temperature=0.3, max_tokens=1024),
    ),
    postprocess=lambda row: dict(
        cat_desc=row["generated_text"],   # the model's extended blurb
        item_id=row["item_id"],
        desc=row["desc"],
    ),
)

In [ ]:
ds = ray.data.read_parquet('/mnt/cluster_storage/scratch/tiny_sample/').repartition(4)
ds = processor(ds)
ds.write_parquet('/mnt/cluster_storage/scratch/tiny_sample_vllm_runai', mode=ray.data.SaveMode.OVERWRITE)

In [ ]:
[(item['desc'], item['cat_desc']) for item in ray.data.read_parquet('/mnt/cluster_storage/scratch/tiny_sample_vllm_runai').take(5)]